Лаб.раб №8.

In [72]:
import copy
from shutil import rmtree
import random
import collections
import numpy as np
import gym
from gym.wrappers import RecordVideo
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
from IPython.display import HTML
from base64 import b64encode
import chex
import warnings
warnings.filterwarnings('ignore')
import gymnasium as gym
from collections import namedtuple

**Упражнение 1:**

In [69]:
def linear_policy(params, obs):
    dot_product_result = jnp.dot(params, obs)
    action = jax.lax.select(dot_product_result > 0, 1, 0)
    return action

# Проверка
fixed_obs = jnp.array([1.0, 1.0, 2.0, 4.0])
params1 = jnp.array([1.0, 1.0, 1.0, 1.0])
params2 = jnp.array([-1.0, -1.0, -1.0, -1.0])
res1 = linear_policy(params1, fixed_obs)
res2 = linear_policy(params2, fixed_obs)
if res1 == 1 and res2 == 0:
    print("✓ Correct")
else:
    print(f"✗ Wrong: {res1}, {res2}")

✓ Correct


**Упражнение 2:**

In [70]:
def run_episode(env):
    episode_return = 0
    terminated = False
    truncated = False
    params = jnp.array([1.0, -2.0, 2.0, -1.0])
    obs, info = env.reset(seed=42)
    while not (terminated or truncated):
        action = linear_policy(params, obs)
        action = int(action)
        obs, reward, terminated, truncated, info = env.step(action)
        episode_return = episode_return + reward
    return episode_return

# Проверка
test_env = gym.make("CartPole-v0")
result = run_episode(test_env)
print(f"Result: {result}")
if result == 31:
    print("✓ Correct")
else:
    print(f"✗ Got {result}, expected 31")
test_env.close()

Result: 31.0
✓ Correct


**Упражнение 3:**

In [74]:
RandomPolicySearchParams = namedtuple("RandomPolicySearchParams", ["current", "best"])
def random_policy_search_choose_action(key, params, actor_state, obs, evaluation=False):
    best_action = linear_policy(params.best, obs)
    current_action = linear_policy(params.current, obs)
    action = jax.lax.select(evaluation, best_action, current_action)
    return action, actor_state

# Проверка
obs = np.ones(4)
current_params = np.ones(4) * -1
best_params = np.ones(4)
rps_params = RandomPolicySearchParams(current_params, best_params)
action1, _ = random_policy_search_choose_action(None, rps_params, None, obs, evaluation=False)
action2, _ = random_policy_search_choose_action(None, rps_params, None, obs, evaluation=True)
print(f"Action without evaluation: {action1}")
print(f"Action with evaluation: {action2}")
if action1 == 0 and action2 == 1:
    print("✓ Correct!")
else:
    print(f"✗ Wrong: got {action1} and {action2}, expected 0 and 1")

Action without evaluation: 0
Action with evaluation: 1
✓ Correct!


**Упражнение 4:**

In [78]:
def get_new_random_weights(random_key, old_weights, minval=-2.0, maxval=2.0):
    new_params = jax.random.uniform(
        random_key,
        shape=old_weights.shape,
        dtype=old_weights.dtype,
        minval=minval,
        maxval=maxval
    )
    return new_params

# Проверка
old_weights = np.ones(4, dtype=np.float32)
random_key = jax.random.PRNGKey(42)
new_weights = get_new_random_weights(random_key, old_weights, -2.0, 2.0)
print(f"Got: {new_weights}")
if np.all(new_weights >= -2.0) and np.all(new_weights <= 2.0):
    print("✓ Correct! Все значения в диапазоне [-2, 2]")
else:
    print("✗ Wrong: значения вне диапазона [-2, 2]")

Got: [-0.04516172  0.7191887   0.46508598  0.24406433]
✓ Correct! Все значения в диапазоне [-2, 2]


**Упражнение 5:**

In [81]:
RandomPolicyLearnState = namedtuple("RandomPolicyLearnState", ["best_average_episode_return"])

def random_policy_search_learn(key, params, learn_state, memory):
    best_params = params.best
    current_params = params.current
    current_average_episode_return = memory
    best_average_episode_return = learn_state.best_average_episode_return
    best_params = jax.lax.select(
        current_average_episode_return > best_average_episode_return,
        current_params,
        best_params
    )
    best_average_episode_return = jax.lax.select(
        current_average_episode_return > best_average_episode_return,
        current_average_episode_return,
        best_average_episode_return
    )
    new_params = get_new_random_weights(key, current_params)
    params = RandomPolicySearchParams(current=new_params, best=best_params)
    return params, RandomPolicyLearnState(best_average_episode_return)

# Проверка
params = RandomPolicySearchParams(
    np.ones(4, dtype=np.float32),
    np.ones(4, dtype=np.float32) * -1
)
learn_state = RandomPolicyLearnState(10)
memory = 11
key = jax.random.PRNGKey(42)
new_params, new_learn_state = random_policy_search_learn(key, params, learn_state, memory)
print(f"Current params: {new_params.current}")
print(f"Best params: {new_params.best}")
print(f"Best average episode return: {new_learn_state.best_average_episode_return}")
if new_learn_state.best_average_episode_return == 11:
    print("✓ Correct! Best average episode return обновился")
else:
    print(f"✗ Wrong: best_average_episode_return = {new_learn_state.best_average_episode_return}, expected 11")
if np.allclose(new_params.best, np.ones(4)):
    print("✓ Correct! Best params не изменились")
else:
    print(f"✗ Wrong: best params изменились")
if not np.allclose(new_params.current, np.ones(4)):
    print("✓ Correct! Current params обновились на случайные")
else:
    print("✗ Wrong: current params не обновились")

Current params: [-0.04516172  0.7191887   0.46508598  0.24406433]
Best params: [1. 1. 1. 1.]
Best average episode return: 11
✓ Correct! Best average episode return обновился
✓ Correct! Best params не изменились
✓ Correct! Current params обновились на случайные


**Упражнение 6:**

In [84]:
def compute_weighted_log_prob(action_prob, episode_return):
    log_prob = jnp.log(action_prob)
    weighted_log_prob = log_prob * episode_return
    return weighted_log_prob

# Проверка
result = compute_weighted_log_prob(0.8, 100)
print(f"Result: {result}")
if result < 0:
    print("✓ Correct! Результат отрицательный (как и должно быть)")
else:
    print(f"✗ Wrong: got {result}")

Result: -22.314353942871094
✓ Correct! Результат отрицательный (как и должно быть)


**Упражнение 7:**

In [85]:
def compute_rewards_to_go(rewards):
    rewards_to_go = []
    total = 0
    for r in reversed(rewards):
        total += r
        rewards_to_go.insert(0, total)
    return rewards_to_go

# Проверка
result = compute_rewards_to_go([1, 2, 3, 4])
print(f"Rewards: [1, 2, 3, 4]")
print(f"Rewards-to-go: {result}")
if result == [10, 9, 7, 4]:
    print("✓ Correct!")
else:
    print(f"✗ Wrong: expected [10, 9, 7, 4]")

Rewards: [1, 2, 3, 4]
Rewards-to-go: [10, 9, 7, 4]
✓ Correct!


**Упражнение 8:**

In [86]:
def sample_action(random_key, logits):
    action = jax.random.categorical(random_key, logits)
    return action

# Проверка
random_key = jax.random.PRNGKey(42)
logits = np.array([1.0, 2.0])
action = sample_action(random_key, logits)
print(f"Logits: {logits}")
print(f"Action: {action}")
if action == 0 or action == 1:
    print("✓ Correct! Действие в допустимом диапазоне [0, 1]")
else:
    print(f"✗ Wrong: action={action}, expected 0 или 1")

Logits: [1. 2.]
Action: 1
✓ Correct! Действие в допустимом диапазоне [0, 1]


**Упражнение 9:**

In [87]:
def policy_gradient_loss(action, logits, reward_to_go):
    all_action_probs = jax.nn.softmax(logits)
    action_prob = all_action_probs[action]
    weighted_log_prob = compute_weighted_log_prob(action_prob, reward_to_go)
    loss = -weighted_log_prob
    return loss

# Проверка
result = policy_gradient_loss(1, np.array([1.0, 2.0]), 10)
print(f"Loss: {result}")

if result > 0:
    print("✓ Correct! Loss положительный (как и должно быть)")
else:
    print(f"✗ Wrong: loss = {result}, expected > 0")

Loss: 3.1326165199279785
✓ Correct! Loss положительный (как и должно быть)


**Упражнение 10:**

In [88]:
def select_greedy_action(q_values):
    action = jnp.argmax(q_values)
    return action

# Проверка
q_values = jnp.array([1.0, 1.0, 3.0, 4.0])
action = select_greedy_action(q_values)
print(f"Q-values: {q_values}")
print(f"Greedy action: {action}")

if action == 3:
    print("✓ Correct!")
else:
    print(f"✗ Wrong: got {action}, expected 3")

Q-values: [1. 1. 3. 4.]
Greedy action: 3
✓ Correct!


**Упражнение 11:**

In [89]:
def compute_squared_error(pred, target):
    squared_error = jnp.square(pred - target)
    return squared_error

# Проверка
result = compute_squared_error(1.0, 4.0)
print(f"Pred=1, Target=4, Squared error={result}")
if result == 9.0:
    print("✓ Correct!")
else:
    print(f"✗ Wrong: got {result}, expected 9.0")

Pred=1, Target=4, Squared error=9.0
✓ Correct!


**Упражнение 12:**

In [90]:
def compute_bellman_target(reward, done, next_q_values):
    bellman_target = jax.lax.select(
        done == 1.0,
        reward,
        reward + jnp.max(next_q_values)
    )
    return bellman_target

# Проверка
res1 = compute_bellman_target(1.0, 0.0, np.array([3.0, 2.0]))
res2 = compute_bellman_target(1.0, 1.0, np.array([3.0, 2.0]))
print(f"Not done (should be 1 + max(3,2)=4): {res1}")
print(f"Done (should be just reward=1): {res2}")
if res1 == 4.0 and res2 == 1.0:
    print("✓ Correct!")
else:
    print(f"✗ Wrong: got {res1} and {res2}")

Not done (should be 1 + max(3,2)=4): 4.0
Done (should be just reward=1): 1.0
✓ Correct!


**Упражнение 13:**

In [91]:
def q_learning_loss(q_values, action, reward, done, next_q_values):
    chosen_action_q_value = q_values[action]
    bellman_target = compute_bellman_target(reward, done, next_q_values)
    squared_error = compute_squared_error(chosen_action_q_value, bellman_target)
    return squared_error

# Проверка
result = q_learning_loss(np.array([3.0, 2.0]), 1, 2.0, 0.0, np.array([3.0, 2.0]))
print(f"Q-loss: {result}")
if result == 9.0:
    print("✓ Correct!")
else:
    print(f"✗ Wrong: got {result}, expected 9.0")

Q-loss: 9.0
✓ Correct!


**Упражнение 14:**

In [92]:
def select_random_action(key, num_actions):
    action = jax.random.randint(key, shape=(), minval=0, maxval=num_actions)
    return action

# Проверка
key1 = jax.random.PRNGKey(6)
key2 = jax.random.PRNGKey(1000)
res1 = select_random_action(key1, 2)
res2 = select_random_action(key2, 2)
print(f"Random action with key=6: {res1}")
print(f"Random action with key=1000: {res2}")
if 0 <= res1 <= 1 and 0 <= res2 <= 1:
    print("✓ Correct! Действия в диапазоне [0, 1]")
else:
    print(f"✗ Wrong: действия вне диапазона")

Random action with key=6: 0
Random action with key=1000: 0
✓ Correct! Действия в диапазоне [0, 1]


**Упражнение 15:**

In [93]:
EPSILON_DECAY_TIMESTEPS = 3000
EPSILON_MIN = 0.1

def get_epsilon(num_timesteps):
    epsilon = 1.0 - (num_timesteps / EPSILON_DECAY_TIMESTEPS)
    epsilon = jax.lax.select(epsilon < EPSILON_MIN, EPSILON_MIN, epsilon)
    return epsilon

# Проверка
res1 = get_epsilon(10)
res2 = get_epsilon(3000)
res3 = get_epsilon(5000)
print(f"Epsilon at step 10: {res1}")
print(f"Epsilon at step 3000: {res2}")
print(f"Epsilon at step 5000: {res3}")
if res1 > res2 and res2 == EPSILON_MIN:
    print("✓ Correct! Epsilon уменьшается и не падает ниже 0.1")
else:
    print("✗ Wrong: проверь логику уменьшения epsilon")

Epsilon at step 10: 0.996666669845581
Epsilon at step 3000: 0.10000000149011612
Epsilon at step 5000: 0.10000000149011612
✓ Correct! Epsilon уменьшается и не падает ниже 0.1


**Упражнение 16:**

In [94]:
def select_epsilon_greedy_action(key, q_values, num_timesteps):
    num_actions = len(q_values)
    epsilon = get_epsilon(num_timesteps)
    random_value = jax.random.uniform(key)
    should_explore = random_value < epsilon

    action = jax.lax.select(
        should_explore,
        select_random_action(key, num_actions),
        select_greedy_action(q_values)
    )
    return action

# Проверка
rng = hk.PRNGSequence(jax.random.PRNGKey(42))
dummy_q_values = jnp.array([0.0, 1.0])
actions_greedy = []
for i in range(5):
    action = select_epsilon_greedy_action(next(rng), dummy_q_values, 5000)
    actions_greedy.append(int(action))
actions_explore = []
for i in range(5):
    action = select_epsilon_greedy_action(next(rng), dummy_q_values, 0)
    actions_explore.append(int(action))
print(f"Жадные действия (много шагов): {actions_greedy}")
print(f"Исследовательские действия (мало шагов): {actions_explore}")
if max(actions_greedy) == 1:
    print("✓ Correct! Жадный режим выбирает действие 1")
else:
    print("✗ Wrong: жадный режим не выбирает 1")
if len(set(actions_explore)) > 1 or 0 in actions_explore:
    print("✓ Correct! Исследовательский режим даёт разные действия")
else:
    print("✗ Wrong: исследовательский режим не работает")

Жадные действия (много шагов): [1, 1, 1, 1, 1]
Исследовательские действия (мало шагов): [0, 1, 0, 0, 0]
✓ Correct! Жадный режим выбирает действие 1
✓ Correct! Исследовательский режим даёт разные действия
